### **Multi-Model AI chatbot**

1. Input - Text + Image
2. Output - Text

In [1]:
# Import Dependancies


import base64
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

d:\GENAIBatch\B2_langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading LLM
load_dotenv()

llm = ChatOpenAI(model="gpt-4o",temperature=0.3)

In [3]:
# Instance of Structred output parser

parser = StrOutputParser()

In [4]:
# Define Memory

memory = ConversationBufferMemory(return_messages=False)
memory

C:\Users\anant\AppData\Local\Temp\ipykernel_34244\2336787887.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=False)


ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]))

In [5]:
# Encode Image

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

Base64 is a way to convert binary data (like images, files, audio) into a text format so it can be safely sent or stored in systems that only handle text.

In [6]:
d = encode_image("D:\\GENAIBatch\\B2_langchain\\class_notebook\\electronic-circuit-voltage-source-resistor-le.jpeg")
d

'...'

In [7]:
type(d)

str

In [8]:
# Prompt Designing
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant with strong computer vision understanding.

Conversation History:
{history}

Task:
- Analyze the image (if provided)
- Answer user query
- Detect important objects
- Explain reasoning

Return output in this structured format:

Answer:
<your answer>

Reasoning:
<step-by-step reasoning>

Detected Objects:
<comma-separated objects>

User Input:
{input}
""")

In [9]:
# Chain creation

chain = prompt | llm | parser

In [10]:
print("Multi-Modal Chatbot Started!\n")

while True:
    user_text = input("You: ")

    if user_text.lower() == ["exit", "bye","quit"]:
        print("!!! Conversation Ended !!!")
        break

    image_path = input("Enter image path (or press Enter to skip): ")

   
    # Get Memory
    history = memory.load_memory_variables({})["history"]


    # Image Handling
    if image_path.strip():
        base64_img = encode_image(image_path)

        vision_response = llm.invoke([
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_text},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_img}"
                        },
                    },
                ],
            }
        ])

        final_input = f"""
User Query: {user_text}

Image Understanding:
{vision_response.content}
"""
    else:
        final_input = user_text


    # Invoke Chain
    
    response = chain.invoke({
        "history": history,
        "input": final_input
    })

 
    # Save to Memory
   
    memory.save_context(
        {"input": user_text},
        {"output": response}
    )

    
    # Output
    print("\nBot Response:\n")
    print(response) 

Multi-Modal Chatbot Started!


Bot Response:

Answer:
The image depicts a basic electrical circuit diagram designed to power an LED. It consists of a voltage source, a resistor, an LED, and a ground connection. The resistor is used to limit the current to a safe level for the LED, ensuring it operates without damage.

Reasoning:
1. **V1** is identified as a voltage source, which is essential for providing the necessary electrical power to the circuit.
2. **R1** is a resistor, a common component used to control the amount of current flowing through the circuit, protecting sensitive components like LEDs from excessive current.
3. **LED1** is a light-emitting diode, which is designed to emit light when an electrical current passes through it. It is a key component in the circuit, as the primary purpose is to light the LED.
4. **GND** represents the ground, which acts as a reference point for the circuit's voltage levels and is crucial for completing the circuit.
5. The arrow labeled **I (

FileNotFoundError: [Errno 2] No such file or directory: 'exit'